<a href="https://colab.research.google.com/github/josephwang02/AAI2025/blob/main/Self_reflecting_Prompt_for_Improving_Output.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""Beginner-friendly recipe suggestions using an in-code recipe collection."""

from __future__ import annotations

import re
from dataclasses import dataclass


@dataclass(frozen=True)
class Recipe:
    name: str
    required: frozenset[str]
    optional: frozenset[str]
    ingredients: tuple[str, ...]
    steps: tuple[str, ...]


RECIPES = (
    Recipe(
        "Cheese Omelet",
        frozenset({"egg", "cheese"}),
        frozenset({"butter", "milk", "salt", "pepper"}),
        ("2 eggs", "1/4 cup shredded cheese", "1 teaspoon butter", "Salt and pepper"),
        (
            "Crack the eggs into a bowl and beat them with a fork.",
            "Melt the butter in a nonstick pan over medium heat.",
            "Pour in the eggs and gently move the pan so the egg spreads out.",
            "When the top is almost set, sprinkle the cheese over one half.",
            "Fold the other half over the cheese.",
            "Cook for about 1 minute, then slide the omelet onto a plate.",
        ),
    ),
    Recipe(
        "Garlic Tomato Pasta",
        frozenset({"pasta", "tomato", "garlic"}),
        frozenset({"olive oil", "parmesan", "salt", "pepper", "basil"}),
        ("8 ounces pasta", "1 cup chopped tomatoes", "2 garlic cloves", "1 tablespoon olive oil"),
        (
            "Boil salted water and cook the pasta according to the package directions.",
            "While it cooks, warm the olive oil in a pan over medium heat.",
            "Add the garlic and cook for about 30 seconds.",
            "Add the tomatoes, salt, and pepper, then cook for 5 minutes.",
            "Drain the pasta, saving a small cup of pasta water.",
            "Toss the pasta with the tomato mixture, adding pasta water if it seems dry.",
            "Top with Parmesan or basil if you have them, then serve.",
        ),
    ),
    Recipe(
        "Vegetable Fried Rice",
        frozenset({"rice", "egg", "vegetable"}),
        frozenset({"soy sauce", "oil", "onion", "garlic", "carrot", "peas"}),
        ("2 cups cooked rice", "1 egg", "1 cup chopped vegetables", "1 tablespoon soy sauce"),
        (
            "Heat oil in a large pan over medium-high heat.",
            "Add the vegetables and cook until they begin to soften.",
            "Move the vegetables aside and scramble the egg in the same pan.",
            "Add the cooked rice and stir everything together.",
            "Pour in the soy sauce and stir until the rice is hot.",
            "Taste carefully and add more soy sauce if needed.",
        ),
    ),
    Recipe(
        "Bean and Cheese Quesadilla",
        frozenset({"tortilla", "cheese", "bean"}),
        frozenset({"oil", "salsa", "onion", "pepper", "corn"}),
        ("1 large tortilla", "1/2 cup beans", "1/2 cup shredded cheese", "Salsa for serving"),
        (
            "Spread the beans over half of the tortilla.",
            "Sprinkle the cheese over the beans and fold the tortilla in half.",
            "Warm a pan over medium heat, with a little oil if needed.",
            "Cook for 2 to 3 minutes on each side.",
            "When golden and melted, cut into wedges and serve with salsa.",
        ),
    ),
    Recipe(
        "Simple Pancakes",
        frozenset({"flour", "egg", "milk"}),
        frozenset({"sugar", "butter", "baking powder", "salt", "banana"}),
        ("1 cup flour", "1 egg", "3/4 cup milk", "1 teaspoon baking powder", "Butter or oil"),
        (
            "Mix the flour, baking powder, and a pinch of salt.",
            "Add the egg and milk, then stir just until mixed.",
            "Heat a lightly greased pan over medium heat.",
            "Pour in a small scoop of batter.",
            "Cook until bubbles form, then flip and cook the other side.",
            "Repeat with the remaining batter and serve warm.",
        ),
    ),
    Recipe(
        "Grilled Cheese Sandwich",
        frozenset({"bread", "cheese"}),
        frozenset({"butter", "tomato", "onion"}),
        ("2 slices of bread", "2 slices of cheese", "1 teaspoon butter"),
        (
            "Put the cheese between the slices of bread.",
            "Spread butter on the outside of each slice.",
            "Heat a pan over medium-low heat.",
            "Cook until the bottom is golden brown.",
            "Flip and cook until the cheese melts and the other side is golden.",
            "Let it cool for a minute, then cut and serve.",
        ),
    ),
)


ALIASES = {
    "eggs": "egg", "tomatoes": "tomato", "vegetables": "vegetable",
    "veggies": "vegetable", "beans": "bean", "tortillas": "tortilla",
    "noodles": "pasta", "cooked rice": "rice", "shredded cheese": "cheese",
}

INGREDIENT_WORDS = {
    "egg": "egg",
    "cheese": "cheese",
    "tomato": "tomato",
    "garlic": "garlic",
    "vegetable": "vegetable",
    "veggie": "vegetable",
    "pasta": "pasta",
    "noodle": "pasta",
    "rice": "rice",
    "bean": "bean",
    "tortilla": "tortilla",
    "flour": "flour",
    "milk": "milk",
    "bread": "bread",
    "butter": "butter",
    "olive oil": "olive oil",
    "oil": "oil",
    "soy sauce": "soy sauce",
    "salt": "salt",
    "pepper": "pepper",
}


def normalize_ingredients(text: str) -> set[str]:
    """Turn a comma-separated list into search terms, ignoring quantities."""
    result = set()
    for item in text.lower().split(","):
        item = re.sub(r"\d+(?:/\d+)?", " ", item)
        item = " ".join(item.strip().split())
        if item:
            result.add(ALIASES.get(item, item))
            for phrase, canonical in INGREDIENT_WORDS.items():
                if phrase in item:
                    result.add(canonical)
    return result


def find_recipes(available: set[str], limit: int = 3) -> list[tuple[Recipe, set[str]]]:
    """Rank recipes by required and optional ingredient matches."""
    matches = []
    for recipe in RECIPES:
        required_matches = available & recipe.required
        optional_matches = available & recipe.optional
        if required_matches:
            score = len(required_matches) * 3 + len(optional_matches)
            missing = recipe.required - available
            matches.append((score, len(missing), recipe.name, recipe, missing))
    matches.sort(key=lambda item: (-item[0], item[1], item[2]))
    return [(item[3], item[4]) for item in matches[:limit]]


def print_recipe(recipe: Recipe, missing: set[str]) -> None:
    """Print a recipe with simple instructions capped at 10 steps."""
    print(f"\n{recipe.name}")
    print("Ingredients:")
    for ingredient in recipe.ingredients:
        print(f"  - {ingredient}")
    matched = len(recipe.required) - len(missing)
    print(f"Main ingredient match: {matched}/{len(recipe.required)}")
    if missing:
        print(f"You may still need: {', '.join(sorted(missing))}")
    else:
        print("You have the main ingredients for this recipe.")
    print("Instructions:")
    for number, step in enumerate(recipe.steps[:10], 1):
        print(f"  {number}. {step}")


def run_recipe_assistant() -> None:
    """Run the interactive recipe search."""
    print("Recipe Assistant")
    print("Enter ingredients separated by commas, such as eggs, cheese, and bread.")
    available = normalize_ingredients(input("What ingredients do you have? "))
    if not available:
        print("Please enter at least one ingredient.")
        return

    matches = find_recipes(available)
    if not matches:
        print("I could not find a close recipe match. Try adding more ingredients.")
        return
    print(f"\nI found {len(matches)} recipe idea(s):")
    for recipe, missing in matches:
        print_recipe(recipe, missing)


if __name__ == "__main__":
    run_recipe_assistant()


Recipe Assistant
Enter ingredients separated by commas, such as eggs, cheese, and bread.
What ingredients do you have? eggs, butter, milk, flour

I found 3 recipe idea(s):

Simple Pancakes
Ingredients:
  - 1 cup flour
  - 1 egg
  - 3/4 cup milk
  - 1 teaspoon baking powder
  - Butter or oil
Main ingredient match: 3/3
You have the main ingredients for this recipe.
Instructions:
  1. Mix the flour, baking powder, and a pinch of salt.
  2. Add the egg and milk, then stir just until mixed.
  3. Heat a lightly greased pan over medium heat.
  4. Pour in a small scoop of batter.
  5. Cook until bubbles form, then flip and cook the other side.
  6. Repeat with the remaining batter and serve warm.

Cheese Omelet
Ingredients:
  - 2 eggs
  - 1/4 cup shredded cheese
  - 1 teaspoon butter
  - Salt and pepper
Main ingredient match: 1/2
You may still need: cheese
Instructions:
  1. Crack the eggs into a bowl and beat them with a fork.
  2. Melt the butter in a nonstick pan over medium heat.
  3. Pour